# Monocrop calculation

This a simulation workflow example to calculate the surfacic density of energy absorbed Eabs (m-2) or the surfacic density of energy incoming Ei (m-2).

Workflow:
1. Select rapeseed variety or Fababean
2. Read climate file, set the sowing dates, set mean temperature data for each species
3. Read Total Leaf Area data (TLA) from file
4. Read parameters based on option (default or specific to an experimental treatment)
5. run simulation over the whole clim data
6. Display 3D (openalea.widgets need to be installed)

In [1]:
import sys
import pandas

from pathlib import Path

from openalea.colzette.colzette import compute_thermal_time, df_to_dict
from openalea.colzette.simulation import run_static_simulation
from openalea.colzette.data import data_dir


## Define identifiers for simulation

In [2]:
Type_simul = 'monocrop_vigo' # 'monocrop_faba', 'monocrop_vigo', 'monocrop_aviso'
if Type_simul == 'monocrop_faba':
  species = 'Fababean' 
else:
  species = 'Rapeseed'

## Read climate file for the chosen simulation site

In [3]:
meteo_fn = data_dir / 'climate' / 'IDEEV_2023_INRAE_STATION_91272001.csv'
clim= pandas.read_csv(meteo_fn,sep=";",skiprows=11)
clim['TM'] = (clim['TN'] + clim['TX'])/2

idx_das = clim.index[
    (clim["AN"] == 2023) &
    (clim["MOIS"] == 9) &
    (clim["JOUR"] == 20)][0] # rapeseed sowing date 14/09/2023
clim2 = clim.iloc[idx_das:]
clim2['TT'] = compute_thermal_time(vec_temp=clim2['TM'], idx_begin=0, Tb=5)

## Read Total Leaf Area data (TLA) for the chosen simulation site

In [4]:
TLA_fn = data_dir / 'input_leaf_surface' / 'set_TLA_IDEEV.csv'
TLA_all = pandas.read_csv(TLA_fn,sep="\t")
TLA_indices_keep = TLA_all[TLA_all['Type']==Type_simul].index
vec_TLA = TLA_all['sim'].values[TLA_indices_keep]

## Read parameters based on option (default or specific to an experimental treatment)

In [5]:
dict_params = df_to_dict(data_dir,Type_simul,Type_simul,"")
density=33.0
option_plants = 'plot' # 'plot' or 'single' i.e. crop or one plant

## run simulation over the whole clim data

In [6]:
dfs = []
list_scene = []
list_cs = []
list_raw = [] 
list_agg = []

for iday in range(0,len(clim2)):

    # just to display running activity
    sys.stdout.write('\r')
    sys.stdout.write(f'{iday/(len(clim2)-1)*100:.0f}' + '%')  # pad to overwrite leftovers
    sys.stdout.flush()

    PlantAge = clim2.iloc[iday]['TT']
    RG_daily = clim2.iloc[iday]['PAR'] * 100.0 # in J.cm-2 must be J.m-2
    TLA = vec_TLA[iday]
    scene, caribu_scene, sub_dat, raw, agg = run_static_simulation(iday,
                                  PlantAge,
                                  RG_daily,
                                  TLA,
                                  option_plants,
                                  density,
                                  dict_params,
                                  type_simul=Type_simul,
                                  species=species, ustride=9, vstride=2, light_direction=(0,0,-1))
    dfs.append(sub_dat)
    list_scene.append(scene)
    list_cs.append(caribu_scene)
    list_raw.append(raw)
    list_agg.append(agg)

res = pandas.concat(dfs,ignore_index=True)

100%

Write results in a file

In [7]:
res.to_csv(f'res_Eabs_{Type_simul}.csv')

## Display 3D

`openalea.widgets` need to be installed

In the second cell below the day after sowing can be changed to visualize the crop at different age.

In [8]:
from openalea.widgets.plantgl import PlantGL

In [9]:
das = len(clim2) - 1 # day after sowing to display

prop = list_agg[das]['Eabs'] # propertie to display: 'Eabs' surfacic absorbed energy or 'Ei' incident energy, list_agg aggregated value entire leaf, list_raw per triangle

scene, values = list_cs[das].plot(prop, display = False)

In [10]:
PlantGL(scene)

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…